# 非線形の候補制約：円形領域・等式・候補間距離

**既知の入力制約**を満たしながら獲得関数を最大化する方法を学びます。円の内部を許す不等式、円周上の等式、2候補の最小距離を要求する候補間制約を、既存の `CandidateConstraints` と `optimize_acqf` で扱います。

制約付き候補提案 → 観測 → `fit()` によるGP再学習 → 次の制約付き候補提案まで実行します。未知の出力制約との違いはNotebook 41を参照してください。


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement

from robotorchan.models import SingleTaskGP
from robotorchan.optim import CandidateConstraints, optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(21)

### 既知の入力制約と未知の出力制約

**候補制約**は入力 `X` の値だけで判定できる条件です。装置の設定範囲や複数の試験条件の距離などが該当します。robotorchanの非線形候補不等式では、**`g(X) >= 0` が可行**です。

一方、試験後にしか分からない品質制約などは未知の出力制約です。こちらはGPで制約を予測するOutcome-constrained BO（Notebook 41）で扱います。

## 2. Fit the surrogate

2次元の既知関数を観測したものとして `SingleTaskGP` を学習します。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    x0 = X[..., :1]
    x1 = X[..., 1:2]
    return 1.0 - 4.0 * (x0 - 0.72).square() - 3.0 * (x1 - 0.58).square()


train_X = torch.rand(16, 2)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

acquisition = qLogExpectedImprovement(model=model, best_f=train_Y.max())
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

### なぜGPと獲得関数を先に作るのか

GPは入力から応答の予測分布を作り、`qLogExpectedImprovement` は改善の期待値を表します。候補制約はその**獲得関数を最大化する際の探索範囲**を制限します。

制約を満たすためにGPの学習データを削除したり、目的関数の定義を変えたりする必要はありません。

## 3. Intra-point nonlinear constraint

中心 `(0.5, 0.5)`、半径 `0.35` の円内だけをcandidateとして許可します。BoTorch/robotorchanのcandidate constraint conventionは `g(X) >= 0` です。

### 円形の可行領域と実行可能な初期点

円の中心を `(0.5,0.5)`、半径を0.35とします。`g(x)=radius²-||x-center||²` は円内で0以上、円外で負になります。

`is_intrapoint=True` の関数には**1候補の `[d]`** が渡されます。非線形制約付きの局所最適化では、制約を満たす `batch_initial_conditions` を明示的に用意します。

In [ ]:
center = torch.tensor([0.5, 0.5])
radius = 0.35


def inside_circle(x: torch.Tensor) -> torch.Tensor:
    return x.new_tensor(radius**2) - (x - center.to(x)).square().sum()


constraints = CandidateConstraints(
    nonlinear_inequality_constraints=((inside_circle, True),),
)

initial_conditions = torch.tensor(
    [
        [[0.50, 0.50]],
        [[0.60, 0.50]],
        [[0.45, 0.60]],
        [[0.55, 0.40]],
    ]
)

candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=None,
    batch_initial_conditions=initial_conditions,
    constraints=constraints,
)

feasibility = inside_circle(candidate[0])
assert feasibility >= -1e-6
print("candidate:", candidate)
print("acquisition value:", value)
print("g(candidate):", feasibility)

### 可行領域を図で確認する

最適化器が返した候補は円内にあるはずです。図には初期観測点と制約境界、提案候補を表示します。**初期観測点が可行領域外にあっても、候補制約は次に提案する点に適用される**ことに注意してください。

## 4. Visualize the feasible region and candidate

In [ ]:
theta = torch.linspace(0.0, 2.0 * torch.pi, 200)
circle_x = center[0] + radius * torch.cos(theta)
circle_y = center[1] + radius * torch.sin(theta)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], label="observed")
ax.plot(circle_x, circle_y, label="g(X) = 0")
ax.scatter(candidate[:, 0], candidate[:, 1], marker="*", s=140, label="candidate")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1")
ax.legend()
plt.show()

### 非線形等式は許容誤差付きで扱う

円周上の条件 `h(x)=0` を厳密な浮動小数点一致で課すのは数値的に難しいため、`|h(x)| <= epsilon` を満たす候補を求めます。

ここでは `epsilon=1e-4` です。最適化後に残差を確認し、指定した許容誤差に収まるか検証します。

## 4a. Intra-point nonlinear equality constraint

非線形等式 `h(x) = 0` を、許容誤差 `epsilon` の範囲で満たす候補を提案します。
BoTorchには非線形等式の直接指定がないため、robotorchanは `|h(x)| <= epsilon` を
2つの非線形不等式へ変換します。微分可能な制約関数と、実行可能な初期点を指定します。

In [ ]:
equality_center = torch.tensor([0.5, 0.5])
equality_radius = 0.25
equality_tolerance = 1e-4


def on_circle(x: torch.Tensor) -> torch.Tensor:
    return (x - equality_center.to(x)).square().sum() - equality_radius**2


equality_constraints = CandidateConstraints(
    nonlinear_equality_constraints=((on_circle, True),),
    nonlinear_equality_tolerance=equality_tolerance,
)
equality_initial_conditions = torch.tensor(
    [
        [[0.75, 0.50]],
        [[0.25, 0.50]],
        [[0.50, 0.75]],
        [[0.50, 0.25]],
    ]
)

equality_candidate, equality_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=None,
    constraints=equality_constraints,
    batch_initial_conditions=equality_initial_conditions,
)

residual = on_circle(equality_candidate[0]).abs()
assert residual <= equality_tolerance + 1e-6
print("candidate:", equality_candidate, "value:", equality_value)
print("nonlinear equality residual:", residual.item())

### 等式制約と不等式制約の違い

円内の不等式では中心も可行ですが、円周上の等式では中心は不可行です。制約の形に合わせて初期点も変更する必要があります。

等式制約の初期値には円周上の4点を指定し、`on_circle(candidate)` の絶対値で制約残差を確認します。

## 5. Joint q-batch with an inter-point constraint

次は `q=2` を同時最適化し、2候補間の二乗距離を `0.12**2` 以上にします。`is_intrapoint=False` なのでcallableはjoint `[q, d]` を受け取ります。

### 候補間制約にはjoint q-batchが必要

2候補を同時に提案する場合、候補間の最小距離を要求することがあります。`is_intrapoint=False` の関数は**候補全体の `[q,d]`** を受け取り、2候補の二乗距離が `min_distance²` 以上かを判定します。

この条件は候補1点だけでは評価できません。`q=1` の最適化を2回独立に行う方法とは異なります。

In [ ]:
min_distance = 0.12


def separated(X: torch.Tensor) -> torch.Tensor:
    squared_distance = (X[0] - X[1]).square().sum()
    return squared_distance - X.new_tensor(min_distance**2)


batch_constraints = CandidateConstraints(
    nonlinear_inequality_constraints=(
        (inside_circle, True),
        (separated, False),
    ),
)

batch_initial_conditions = torch.tensor(
    [
        [[0.40, 0.50], [0.60, 0.50]],
        [[0.50, 0.38], [0.50, 0.62]],
        [[0.38, 0.42], [0.62, 0.58]],
        [[0.42, 0.62], [0.58, 0.38]],
    ]
)

batch_candidate, batch_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=None,
    batch_initial_conditions=batch_initial_conditions,
    constraints=batch_constraints,
)

assert all(inside_circle(x) >= -1e-6 for x in batch_candidate)
assert separated(batch_candidate) >= -1e-6
print("q=2 candidates:\n", batch_candidate)
print("joint acquisition value:", batch_value)
print("inter-point g(X):", separated(batch_candidate))

### 2候補が同時に可行であることを確認する

2候補がそれぞれ円内にあり、かつ互いに一定以上離れている必要があります。図では2候補を破線で結び、距離制約がどの2点に作用するか示します。

## 6. Visualize the constrained q-batch

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], label="observed")
ax.plot(circle_x, circle_y, label="intra-point boundary")
ax.scatter(
    batch_candidate[:, 0],
    batch_candidate[:, 1],
    marker="*",
    s=140,
    label="q=2 candidates",
)
ax.plot(batch_candidate[:, 0], batch_candidate[:, 1], linestyle="--")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1")
ax.legend()
plt.show()

## 7. Update the data

candidate constraintは候補生成時の既知feasibilityです。実験・シミュレーション後は通常どおり新しい `(X, Y)` をtraining dataへ追加します。

### 制約を満たした候補を実際に観測する

ここではjoint `q=2` で提案した2点を人工関数 `response` で評価します。既知の入力制約なので、**制約値を新たなGPで学習する必要はありません**。

観測値 `Y` を訓練データへ追加したらGPを再学習し、更新した獲得関数から次の2候補を提案します。

In [ ]:
new_Y = response(batch_candidate)
updated_X = torch.cat([train_X, batch_candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, new_Y.detach()], dim=0)

assert updated_X.shape == torch.Size([18, 2])
assert updated_Y.shape == torch.Size([18, 1])
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 8. 再学習後に制約付きq=2候補を再提案する

更新したGPで `fit()` を実行し、`best_f` を新しい観測最大値へ更新します。最適化時には**同じ円内制約と候補間距離制約**を再利用します。

In [ ]:
updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
updated_acquisition = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_Y.max(),
)
next_batch, next_value = optimize_acqf(
    acq_function=updated_acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=None,
    batch_initial_conditions=batch_initial_conditions.clone(),
    constraints=batch_constraints,
)
assert next_batch.shape == (2, 2)
assert all(inside_circle(x) >= -1e-6 for x in next_batch)
assert separated(next_batch) >= -1e-6
print("Next constrained q=2 candidates:\n", next_batch)
print("Next acquisition value:", next_value)
print("Updated best observed response:", updated_Y.max().item())

### 観測前後のGP予測平均を比較する

入力空間を2次元格子に区切り、観測前後のGP予測平均を比較します。可行領域の境界と新しい2観測点も重ねて表示します。

可行領域は既知の幾何学的条件なので、GPの再学習によって円の境界が変わるわけではありません。

In [ ]:
axis = torch.linspace(0.0, 1.0, 41)
mesh_x0, mesh_x1 = torch.meshgrid(axis, axis, indexing="ij")
plot_X = torch.stack([mesh_x0.reshape(-1), mesh_x1.reshape(-1)], dim=-1)
with torch.no_grad():
    before_mean = model.posterior(plot_X).mean.reshape(41, 41)
    after_mean = updated_model.posterior(plot_X).mean.reshape(41, 41)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
low = min(before_mean.min().item(), after_mean.min().item())
high = max(before_mean.max().item(), after_mean.max().item())
levels = torch.linspace(low, high, 18).numpy()
for ax, surface, title in (
    (axes[0], before_mean, "Before observation"),
    (axes[1], after_mean, "After observation and refit"),
):
    im = ax.contourf(mesh_x0.numpy(), mesh_x1.numpy(), surface.numpy(), levels=levels)
    ax.plot(circle_x, circle_y, label="Feasible boundary")
    ax.scatter(batch_candidate[:, 0], batch_candidate[:, 1], marker="x", s=70, label="New observations")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1", title=title)
    ax.legend(fontsize=8)
fig.colorbar(im, ax=axes, label="GP posterior mean")
plt.show()

## 8. Contract summary

- candidate nonlinear feasibilityは `g(X) >= 0`。
- intra-point callableは `[d]`、inter-point callableはjoint `[q, d]` を受け取ります。
- nonlinear optimizationにはfeasible initial conditionsまたは適切な `ic_generator` が必要です。
- inter-point nonlinear constraintはjoint q semanticsを必要とし、greedy sequentialへ分解できません。
- candidate constraintはOutcome constraint `c(Y) <= 0` と独立して併用できます。
- robust input perturbationでは通常nominal Xを制約し、全perturbation scenarioのfeasibilityを保証するものではありません。

## 10. 公開APIの整理

| 目的 | 既存API | 注意点 |
|---|---|---|
| GPの学習と再学習 | `SingleTaskGP.fit()` | 観測データを追加 |
| 獲得関数 | `qLogExpectedImprovement` | 最大化・観測最良値 `best_f` |
| 既知の非線形候補制約 | `CandidateConstraints(nonlinear_inequality_constraints=...)` | `g(X) >= 0` が可行 |
| 非線形等式 | `nonlinear_equality_constraints` | 許容誤差付き |
| 候補間距離制約 | `is_intrapoint=False` | joint `[q,d]` で判定 |
| 制約付き候補最適化 | `optimize_acqf` | 可行な初期値を指定 |

**重要：** 候補制約は既知の入力条件であり、未知の出力制約とは異なります。候補間制約を独立したq=1最適化へ分解しないでください。